In [26]:
import os
import rootutils
import pandas as pd
import random
import numpy as np

import matplotlib.pyplot as plt

from tqdm.notebook import tqdm
tqdm.pandas()

rootutils.setup_root(os.path.abspath('./'), indicator=".project-root", pythonpath=True, dotenv=True, cwd=True)

# auto-loading of imports from outside scripts
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [27]:
from src.cif_utils import cif_from_file

In [28]:
meta_csv = "data_cod/cod_bradley_merged.csv"
cifs_dir = "cifs"

In [29]:
meta_df = pd.read_csv(meta_csv).drop_duplicates().sort_values(by="id")
meta_df["cif_path"] = cifs_dir + "/" + meta_df["id"].astype(str) + ".cif"

---
## Coordinational Numbers:

In [42]:
from pymatgen.core import Structure
import numpy as np
import pandas as pd
from collections import defaultdict

In [61]:
def is_interior(site, frac_margin):
    ""
    f = site.frac_coords
    return np.all(f > frac_margin) and np.all(f < 1 - frac_margin)

def get_neighbours(cif_path: str, cutoffs: dict):
    structure = Structure.from_file(cif_path)
    
    # To cut edge atoms:
    max_cutoff = max(cutoffs.values())
    frac_margin = max_cutoff / min(structure.lattice.abc)

    # Compute statistics:
    coord_numbers = defaultdict(lambda: defaultdict(list))
    for i, site in enumerate(structure.sites):
        if not is_interior(site, frac_margin):
            continue
        central = site.specie.symbol
        for (e1, e2), cutoff in cutoffs.items():
            if central not in (e1, e2):
                continue
            # Finding all neighbours of the central atom:
            neighs = structure.get_neighbors(site, cutoff)
            count = sum(1 for n in neighs if n.specie.symbol == (e2 if central == e1 else e1))
            coord_numbers[central][(e1, e2)].append(count)

    # Average:
    results = []
    for central, partners in coord_numbers.items():
        for pair, counts in partners.items():
            avg_cn = np.mean(counts)
            results.append({
                "central": central,
                "pair": f"{pair[0]}–{pair[1]}",
                "avg_cn": avg_cn,       # Average coordination number: mean number of neighbors of that type
                "n_sites": len(counts)  # Number of central atoms included in the average
            })

    df = pd.DataFrame(results)
    
    return df

In [63]:
cutoffs = {
    ("O", "H"): 1.27,
    ("O", "C"): 1.72,
    ("C", "H"): 1.37,
}

error_cifs = []
for cif_path in tqdm(meta_df["cif_path"], total=len(meta_df), desc="Processing CIFs"):
    try:
        df = get_neighbours(cif_path, cutoffs)
    except Exception as e:
        error_cifs.append(cif_path)
        
    break


Processing CIFs:   0%|          | 0/13528 [00:00<?, ?it/s]

In [64]:
df

,central,pair,avg_cn,n_sites
0,H,O–H,0.142857,28
1,H,C–H,0.857143,28
2,C,O–C,0.363636,22
3,C,C–H,1.363636,22
4,O,O–H,1.000000,6
5,O,O–C,1.000000,6
